# 구조화 검색 DB 읽기 전용 확인

운영 코드/DB/이전 실험을 변경하지 않고 실제 SQLite 스키마와 저장값을 확인한다. 유료 API·OCR·모델 추론은 호출하지 않는다. 현재 검색과 근거 보완 검색의 성능 비교는 아직 실행하지 않는다.

[상세 명세 Markdown](../docs/STRUCTURED_SEARCH_DB_SPEC.md) · [HTML 명세](../docs/STRUCTURED_SEARCH_DB_SPEC.html)

반복 가능한 DB 검사는 `44_inspect_structured_db.py`가 담당한다. 명세는 docs/의 MD·HTML로 관리하며 전용 생성기는 사용하지 않는다. 이 노트북은 저장 inventory를 입력과 대조하고 주요 명세·검사 결과를 표시한다. 실행 시 CWD가 저장소 루트 또는 notebooks여도 자동 탐색한다.

In [1]:
from pathlib import Path
import importlib.util
import json
from IPython.display import Markdown, display
location = Path.cwd().resolve()
ROOT = next(p for p in [location, *location.parents] if (p / 'notebooks/44_inspect_structured_db.py').exists())
spec = importlib.util.spec_from_file_location('db_spec44', ROOT / 'notebooks/44_inspect_structured_db.py')
inspector = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inspector)
saved = json.loads((inspector.OUT / 'schema_inventory.json').read_text())
fresh = inspector.inventory()
assert fresh == saved, '입력 또는 저장 inventory 변경'
print('읽기 전용 재검사 통과:', {name: t['row_count'] for name, t in fresh['tables'].items()})
print('총 SQL 컬럼:', sum(len(t['columns']) for t in fresh['tables'].values()))

읽기 전용 재검사 통과: {'metadata': 8, 'documents': 106, 'facts': 3403, 'target_aliases': 22}
총 SQL 컬럼: 41


In [2]:
document = (ROOT / 'docs/STRUCTURED_SEARCH_DB_SPEC.md').read_text()
start = document.index('## 3. ')
end = document.index('## 7. ')
display(Markdown(document[start:end]))

## 3. `metadata` 테이블 — 전체 컬럼


타입은 SQLite DDL의 선언 타입이다. 제약은 DB가 강제하는 선언이며, 값 규칙은 빌드/검색 코드의 의미다. enum과 단위·JSON 형식은 CHECK로 강제하지 않는다. 예시는 규칙 설명이며 실제 관측값과 구분한다.


| 컬럼 | 선언 타입 | DDL 제약 | 의미 | 값 규칙·예시 | 현재 검색 용도 |
| --- | --- | --- | --- | --- | --- |
| `key` | TEXT | NOT NULL 선언 없음, PK, DEFAULT 없음 | 빌드 메타데이터 항목 이름 | 현재 8개 key. 아래 key별 표 참조 | 버전·출처·정책 추적 |
| `value_json` | TEXT | NOT NULL, DEFAULT 없음 | 항목 값의 JSON 문자열 | 문자열/객체/목록/숫자 구조를 JSON으로 직렬화 | 검색 조건 자체가 아니라 빌드 설명 |

## 4. `documents` 테이블 — 전체 컬럼


타입은 SQLite DDL의 선언 타입이다. 제약은 DB가 강제하는 선언이며, 값 규칙은 빌드/검색 코드의 의미다. enum과 단위·JSON 형식은 CHECK로 강제하지 않는다. 예시는 규칙 설명이며 실제 관측값과 구분한다.


| 컬럼 | 선언 타입 | DDL 제약 | 의미 | 값 규칙·예시 | 현재 검색 용도 |
| --- | --- | --- | --- | --- | --- |
| `document_id` | TEXT | NOT NULL 선언 없음, PK, DEFAULT 없음 | 문서/카드 식별 키 | 원본 document_id. 예: BC/BC_Green_v2. basename만으로 만들지 않음 | card_keys 및 facts.document_id와 연결 |
| `provider` | TEXT | NOT NULL, DEFAULT 없음 | 원본 OCR 제공자 | 이 빌드는 upstage만 허용. SQL enum 제약은 없음 | 제공자와 출처 구분 |
| `card_name` | TEXT | NOT NULL 선언 없음, DEFAULT 없음 | 원본 identity.card_name | 원문 카드명; 누락 시 NULL 가능 | LLM에 제공할 카드 목록 구성 |
| `issuer_name` | TEXT | NOT NULL 선언 없음, DEFAULT 없음 | 원본 identity.issuer_name | 원문 카드사명; 누락 시 NULL 가능 | 표시/카드 목록 정보. 별도 issuer 조건 없음 |
| `source_pdf_sha256` | TEXT | NOT NULL, DEFAULT 없음 | 원본 PDF 내용 SHA-256 | 원본 JSON에서 복사. DB는 길이·hex 형식을 강제하지 않음 | 원문 동일성 추적 |
| `normalized_json_sha256` | TEXT | NOT NULL, DEFAULT 없음 | 입력 normalized.json bytes SHA-256 | 파일 실제 bytes로 계산. JSON 의미 동일성과는 다름 | 입력 변경 추적 |
| `source_relative_path` | TEXT | NOT NULL, DEFAULT 없음 | normalized.json의 저장소 상대 경로 | 현재 빌드 입력 glob의 문서 경로 | 원문 출처 접근 및 정렬 |
| `source_path_sha256` | TEXT | NOT NULL, DEFAULT 없음 | 위 상대 경로 문자열 SHA-256 | UTF-8 경로를 해시. 파일 내용 해시가 아님 | 경로 동일성 추적 |
| `page_count` | INTEGER | NOT NULL, DEFAULT 없음 | 저장 pages 목록 길이 | 0 이상 정수로 생성. SQL CHECK 없음 | 원문 규모 정보 |
| `fact_count` | INTEGER | NOT NULL, DEFAULT 없음 | 해당 문서 원본 facts 목록 길이 | 일반 안내도 포함. 순수 혜택 수와 다름 | 문서별 fact 수 확인 |
| `raw_json` | TEXT | NOT NULL, DEFAULT 없음 | 문서 보존용 객체의 JSON 문자열 | identity/pages/provenance/해시 등. 원본 normalized.json 전체 bytes 복제는 아님 | load_facts의 원문 페이지·카드 정체성 복원 |

## 5. `facts` 테이블 — 전체 컬럼


타입은 SQLite DDL의 선언 타입이다. 제약은 DB가 강제하는 선언이며, 값 규칙은 빌드/검색 코드의 의미다. enum과 단위·JSON 형식은 CHECK로 강제하지 않는다. 예시는 규칙 설명이며 실제 관측값과 구분한다.


| 컬럼 | 선언 타입 | DDL 제약 | 의미 | 값 규칙·예시 | 현재 검색 용도 |
| --- | --- | --- | --- | --- | --- |
| `fact_id` | TEXT | NOT NULL 선언 없음, PK, DEFAULT 없음 | 문서 내 원본 fact 식별자 | fact_ + SHA-256 앞 24hex. provider&#124;PDF해시&#124;document_id&#124;ordinal로 계산 | matching_fact_ids → 원문 근거 연결 |
| `document_id` | TEXT | NOT NULL, FK → documents.document_id, 두 컬럼 조합 UNIQUE, DEFAULT 없음 | 소유 문서/카드 키 | documents.document_id를 참조 | 같은 카드 안의 조건 평가 및 근거 소유권 |
| `ordinal` | INTEGER | NOT NULL, 두 컬럼 조합 UNIQUE, DEFAULT 없음 | 원본 facts 목록의 0부터 시작하는 순서 | enumerate로 생성. (document_id, ordinal) UNIQUE | 원본 fact 순서 복원. 목록 변경 시 fact_id가 달라질 수 있음 |
| `target_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 대상 문자열 | 가맹점/업종/대상 목록/안내 대상 등 자유 문자열. 미기재는 빈 문자열 가능 | derived.target_normalized에 대상/별칭 포함 여부 비교 |
| `action_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 행동 문자열 | 할인/청구할인/적립/안내/부정 문구 등 자유 문자열 | benefit_type_raw와 함께 action_kind/status 생성 |
| `benefit_type_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 혜택 이름/분류 설명 | 자유 문자열. 고정 카테고리 enum 아님 | 행동 정규화 보조. 별도 분류 필터 없음 |
| `value_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 수치/값 문구 | 5, 3~5, 최대 10 등 문자열. 단일 숫자로 제한하지 않음 | typed ratio와 일치할 때 percent 생성 |
| `unit_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 단위 문자열 | %, 원, 마일리지, 포인트, 빈 문자열 등. 전체 관측값은 inventory 참조 | percent 정규화 보조. 단위 자유 비교 필터 없음 |
| `condition_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 적용 조건 문구 | 전월 실적/결제 방식/복합 조건 등을 원문 형태로 보존 | 명시적인 단일 전월 최소 실적만 파생 비교 |
| `cap_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 한도 문구 | 혜택 지급 한도/이용금액 상한 등 원문 표현 | 지급 한도만 보수적으로 cap_krw 생성 |
| `period_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 기간 문구 | 월/일/지급일/행사 기간 등 자유 문자열 | cap_raw와 합쳐 월/일 한도 기간 식별. 행사 날짜 검색 아님 |
| `frequency_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 횟수 조건 | 월 1회 등 자유 문자열. 미기재는 빈 문자열 | 보존만 함. 횟수 수치 필터 없음 |
| `exceptions_raw` | TEXT | NOT NULL, DEFAULT 없음 | 원본 제외/예외 문구 | 제외 업종/채널/기타 예외 등 자유 문자열 | 요구 대상이 예외에 언급되면 unknown으로 보수 처리 |
| `action_kind` | TEXT | NOT NULL 선언 없음, DEFAULT 없음 | 정규화한 행동 종류 | discount(할인), earn(적립 계열), NULL(판단 불가) | Requirement.action과 비교 |
| `action_status` | TEXT | NOT NULL, DEFAULT 없음 | 행동을 긍정 혜택으로 볼 수 있는지 | true(긍정), false(부정/제외), unknown(혼합/모호) | false 제외; unknown은 확정 충족으로 선택하지 않음 |
| `percent` | REAL | NOT NULL 선언 없음, DEFAULT 없음 | 비율의 퍼센트값 | 10% → 10.0, 0.1이 아님. known일 때 수치, 그 외 NULL | percent >= min_percent |
| `percent_state` | TEXT | NOT NULL, DEFAULT 없음 | 비율값의 사용 가능 상태 | known / unknown / not_applicable. 아래 상태 정의 참조 | 비율 요구 시 known만 확정 비교 |
| `spend_threshold_krw` | INTEGER | NOT NULL 선언 없음, DEFAULT 없음 | 필요 전월 최소 실적 원화 금액 | 30만원 이상 → 300000. known일 때 수치, 그 외 NULL | 필요 실적 <= 질문의 max_spend_krw |
| `spend_operator` | TEXT | NOT NULL 선언 없음, DEFAULT 없음 | 원문 실적 부등호 판정 | gte(이상), lte(이하), unknown, NULL을 코드가 만들 수 있음. 관측값은 별도 표 | known은 gte만. 질문의 최대 허용 실적과 부등호 방향을 구분 |
| `spend_state` | TEXT | NOT NULL, DEFAULT 없음 | 전월 실적값 사용 가능 상태 | known / unknown / not_applicable | 실적 요구 시 known만 확정 비교 |
| `cap_krw` | INTEGER | NOT NULL 선언 없음, DEFAULT 없음 | 혜택 지급 한도 원화 금액 | 월 할인 한도 5천원 → 5000. 이용금액 상한과 구분 | cap_krw >= min_cap_krw |
| `cap_state` | TEXT | NOT NULL, DEFAULT 없음 | 혜택 한도값 사용 가능 상태 | known / unknown / not_applicable | 한도 요구 시 known만 확정 비교 |
| `cap_period` | TEXT | NOT NULL 선언 없음, DEFAULT 없음 | 한도와 연결된 기간 | month(월), day(일), NULL(모호/미기재) | min_cap_krw 요구가 있을 때 cap_period_state와 함께 기간 비교 |
| `raw_json` | TEXT | NOT NULL, DEFAULT 없음 | 원본 fact 객체의 JSON 문자열 | 원문 10필드 + typed_normalization + field_evidence + relation_scope 등 보존 | 검색 load_facts는 이 JSON에서 원문/근거를 복원 |
| `derived_json` | TEXT | NOT NULL, DEFAULT 없음 | 보수적으로 계산한 파생 객체 JSON | SQL 파생 컬럼 + 대상 정규화/사유/기간 상태/예외 정규화 포함 | 현재 선택기는 복원된 derived 객체로 비교 |

## 6. `target_aliases` 테이블 — 전체 컬럼


타입은 SQLite DDL의 선언 타입이다. 제약은 DB가 강제하는 선언이며, 값 규칙은 빌드/검색 코드의 의미다. enum과 단위·JSON 형식은 CHECK로 강제하지 않는다. 예시는 규칙 설명이며 실제 관측값과 구분한다.


| 컬럼 | 선언 타입 | DDL 제약 | 의미 | 값 규칙·예시 | 현재 검색 용도 |
| --- | --- | --- | --- | --- | --- |
| `alias_group` | TEXT | NOT NULL, PK(1), DEFAULT 없음 | 별칭 그룹 이름 | 여행, 커피, 카페. LLM이 동적으로 생성하지 않음 | 개발자가 고정한 대상 확장 규칙 |
| `alias` | TEXT | NOT NULL, PK(2), DEFAULT 없음 | 그룹에 속한 원문 별칭 | 스타벅스, 항공, 호텔 등. (alias_group, alias) 복합 PK | 동일 그룹 내 대상 대안 |
| `alias_normalized` | TEXT | NOT NULL, DEFAULT 없음 | 별칭의 정규화 문자열 | NFKC → casefold → 모든 공백 제거 | 현재 선택기는 코드의 TARGET_ALIASES를 사용; 이 테이블은 규칙 보존용 |



In [3]:
start = document.index('## 7. ')
end = document.index('## 9. ')
display(Markdown(document[start:end]))

## 7. JSON 내부 속성과 출처 연결

아래 속성은 별도 SQL 컬럼이 아니다. DB의 TEXT JSON을 파싱해 접근한다. JSON 유효성/필드 필수 여부는 SQL CHECK로 강제되지 않는다. `_raw` SQL 컬럼을 수정하는 것으로 JSON이 자동 갱신되지 않으며 동기화 trigger도 없다. 이번 작성은 양쪽 3,403행의 일치를 확인했다.

### metadata.value_json: key별 속성값

| key | JSON 값 종류 | 내용 |
| --- | --- | --- |
| schema_version | string | structured-benefit-experiment-v3 |
| source_glob | string | data/rag/runtime/working/run_f2d29820c6dca642/documents/*/upstage/normalized.json |
| source_policy | string | Upstage JSON 보존, OCR/검증 통과 가정 |
| source_version | object | provider, structure_schema_version, normalization_contract, normalization_config_hash, normalization_model_recorded_in_source, normalization_reasoning_recorded_in_source |
| counts | object | documents, pages, facts, known_percent, known_spend_threshold, known_cap, known_cap_period, positive_supported_action |
| support_rates | object | known_*와 positive_supported_action의 전체 facts 대비 비율 |
| target_aliases | object | 여행/커피/카페 → 별칭 목록 |
| limitations | array[string] | OCR 미재검증, 복잡한 수치/기간의 unknown 처리 등 |

외부 db_build_summary.json의 verification/artifact_sha256는 DB metadata에 삽입한 후에 추가한다. 따라서 그 두 key가 metadata에 없다는 것은 누락 오류가 아니다.

### documents.raw_json

| 속성 | 값 구조·역할 |
| --- | --- |
| document_id, provider, *_sha256, source_relative_path, page_count, fact_count | 문서 SQL 값에 대응하는 출처 정보 |
| identity | card_name, issuer_name, card_evidence.line_ids, issuer_evidence.line_ids |
| structure_schema_version, normalization_contract | 원본 구조화/정규화 계약 버전 |
| provenance | 구조화 추출의 model, reasoning, config_hash, prompt_sha256, schema_sha256, endpoint, max_output_tokens 등 원본 기록 |
| ocr_provenance, ocr_parse_provenance | OCR 제공자 설정과 정규화 정책의 원본 기록 |
| pages[] | 원본 정규화 페이지 객체. blocks/text/표 등 원문 연결에 필요한 자료를 보존. 페이지별 속성은 원본 제공 형태에 따름 |

이 객체는 builder가 선택한 문서 정보를 저장한다. 원본 facts는 facts.raw_json에 따로 저장하며 documents.raw_json에 facts 목록을 중복 저장하지 않는다.

### facts.raw_json

| 속성 | 값 구조·예시·용도 |
| --- | --- |
| target/action/benefit_type/value/unit/condition/cap/period/frequency/exceptions | SQL의 원문 10개 *_raw 컬럼에 대응 |
| typed_normalization | 원본의 선택적 정규화 객체. 필드명 → [{kind, decimal}, …], condition_operators → 비교 연산자 목록 등. 현재 raw_json에서 키 누락 25행, 객체 3,378행, 명시적 null 0행. builder/reader의 get(...) or {}는 키 누락 및 null을 {}로 취급. 실제 관측 kind는 ratio, KRW, number |
| typed_normalization.*[].kind | ratio=비율, KRW=원화, number=일반 숫자. 현재 수치 파생기는 value의 ratio, condition/cap의 KRW를 사용 |
| typed_normalization.*[].decimal | 정밀 수치 문자열. 10%의 ratio는 "0.1", 30만원 KRW는 "300000". percent 컬럼의 10.0과 표현 단위가 다름 |
| field_evidence | 원문 필드명 → 근거 fragment 목록. 미기재/근거 없음은 빈 목록 가능 |
| field_evidence.*[].line_id | 원본 줄 레지스트리 참조. 예: P0003-L0010. 검색 chunk_id가 아님 |
| field_evidence.*[].fragment | 해당 줄에서 발췌한 원문. 예: 청구할인 |
| field_evidence.*[].char_start / char_end | 줄 문자열의 0부터 시작하는 시작/끝 위치. Python 슬라이스 [start:end], 끝 제외. [원문 위치 처리 코드](../jobs/rag-indexer/src/pickcardu_indexer/pipeline.py) 참고 |
| relation_scope.scope_type | 실제 sentence(문장), table_row(표 행), bounded_block(제한된 블록) |
| relation_scope.line_ids | 같은 fact를 설명하는 원문 줄 목록 |
| relation_scope.row_line_ids / header_line_ids | 표 행/헤더의 연결 줄 목록. 해당하지 않으면 빈 목록 |

raw_json의 typed_normalization 키는 필수가 아니다. SQL/JSON 보존본과 달리 load_facts 및 facts.jsonl 최상위의 typed_normalization/field_evidence/relation_scope는 값이 누락되거나 null이면 {}로 복원한다. 현재 field_evidence/relation_scope는 원문에서도 모두 객체다. 객체임을 가정한 소비자는 먼저 키 존재와 값을 처리해야 한다.

원문 근거 참조를 보존했다는 사실은 참조/의미의 정확성이 이번에 검증됐다는 뜻이 아니다. 하나의 fact에 여러 줄·필드 fragment가 있을 수 있어 fact_id를 검색 chunk_id 하나로 곧바로 바꿀 수 있다고 가정하지 않는다.

### facts.derived_json: 전체 파생 속성

| 속성 | 값·의미·검색 사용 |
| --- | --- |
| target_normalized | 대상 원문에 NFKC/casefold/공백 제거 적용. 별도 SQL 컬럼 없음 |
| action_kind, action_status | SQL 동명 컬럼과 동일 |
| percent, percent_state | SQL 동명 컬럼과 동일 |
| spend_threshold_krw, spend_operator, spend_state | SQL 동명 컬럼과 동일 |
| cap_krw, cap_state, cap_period | SQL 동명 컬럼과 동일 |
| cap_period_state | known/unknown/not_applicable. **별도 SQL 컬럼 없음**. 한도 기간 필터에서 사용 |
| action_reason / percent_reason / spend_reason / cap_reason / cap_period_reason | 각 값/상태를 만든 이유 문자열. 실제 전체 사유와 빈도는 inventory.nested_value_counts 참조. 판정 확률이나 LLM confidence가 아님 |
| period_raw | 원본 period 문자열의 파생 객체 내 보존본 |
| has_exceptions | 원본 예외 문구가 공백 제외 비어 있지 않으면 true. 현재 선택기가 이 bool을 직접 필터하지 않음 |
| exceptions_normalized | 예외 문구 정규화 문자열. 요구 대상이 포함되면 unknown 처리 |


### JSON 키 누락과 명시적 null의 실제 관측

Missing(키 누락), Null(명시적 null), Empty object(빈 객체 {})를 구분한다. facts 원문/복원값은 각 3,403행, documents는 106행이 분모다. 품질 점수가 아니며 원문 누락값을 복원한 것이 신규 정보 추출을 뜻하지 않는다.


| 위치 | 속성 | 실제 타입: 행 수 | Missing | Null | Empty object |
| --- | --- | --- | --- | --- | --- |
| facts.raw_json | typed_normalization | object: 3378; missing: 25 | 25 | 0 | 0 |
| facts.raw_json | field_evidence | object: 3403 | 0 | 0 | 0 |
| facts.raw_json | relation_scope | object: 3403 | 0 | 0 | 0 |
| reader_restored_facts | typed_normalization | object: 3403 | 0 | 0 | 25 |
| reader_restored_facts | field_evidence | object: 3403 | 0 | 0 | 0 |
| reader_restored_facts | relation_scope | object: 3403 | 0 | 0 | 0 |
| documents.raw_json | identity | object: 106 | 0 | 0 | 0 |
| documents.raw_json | pages | array: 106 | 0 | 0 | 0 |
| documents.raw_json | provenance | object: 106 | 0 | 0 | 0 |
| documents.raw_json | ocr_provenance | object: 106 | 0 | 0 | 0 |
| documents.raw_json | ocr_parse_provenance | object: 106 | 0 | 0 | 0 |

typed_normalization은 원본 raw_json에서 객체 3,378행·키 누락 25행·명시적 null 0행이다. 현재 reader는 누락된 키를 {}로 복원하므로 읽은 facts에는 객체 3,403행이 있다. 원본 저장 형태와 reader 결과를 혼동하지 않는다.


## 8. 정규화 규칙과 질문 스키마의 연결

### 상태값의 의미

| 상태 | 의미 | 예시 | 잘못 해석하면 안 되는 점 |
| --- | --- | --- | --- |
| known | 코드 규칙상 단일 값으로 비교 가능 | 원문 10%와 typed ratio 0.1이 일치 | PDF 사실 정확도 검증 성공을 뜻하지 않음 |
| unknown | 원문에 값/문구가 있으나 모호·복합·불일치·지원 밖 | 최대 10%, 3~5%, 두 실적 구간 | 0/false로 바꾸지 않음 |
| not_applicable | 해당 파생 종류의 값/표현을 확인하지 못한 분기 | value/ratio 없음, cap 없음 등 | 카드에 혜택/실적/한도가 없다는 전역 결론 아님 |

- 비율: 원본 typed ratio 하나 ×100과 원문 퍼센트값 하나가 일치해야 known. 범위/상한/하한/원화 혼합은 unknown. 질문의 최소 비율과 원문의 최대 비율을 혼동하지 않는다.
- 실적: 명시 전월 실적 문구에 연결된 단일 금액·이상(gte)이 typed KRW와 맞아야 known. 코드가 이하(lte)를 식별할 수 있지만 확정 실적값으로 지원하지 않는다. 복수 구간은 unknown.
- 지급 한도: 원문 금액 하나와 typed KRW 하나가 맞아야 known. 이용/결제/승인 금액 상한은 지급 혜택 한도로 사용하지 않는다.
- 한도 기간: cap/period 문구의 월/일을 확인한다. 둘 다 등장하거나 불명확하면 unknown. 일반 period는 지급일·행사 기간일 수도 있다.
- 행동: action + benefit_type에서 할인/적립 계열을 식별한다. 제외/제공되지 않음/적용되지 않음/불가이면 false, 할인과 적립 혼합은 unknown. **질문 지시문은 earn에 캐시백을 포함한다고 설명하지만 현재 정규화 함수의 적립 토큰에는 캐시백이 없다.** 캐시백만 있는 문구가 earn으로 정규화된다고 보장하지 않는다.

### LLM이 출력하는 QueryPlan

LLM은 스키마를 새로 만들거나 SQL을 생성하지 않는다. 고정 Pydantic QueryPlan에 맞춘 JSON 값을 출력한다. 추가 속성은 extra=forbid지만 질문 의미의 정확성은 별도 검사 대상이다. 다음 필드는 기본값이 없어 출력에 필요하며 nullable은 null 허용을 뜻한다.

| 속성 | 타입·허용값 | 처리 |
| --- | --- | --- |
| route | unstructured / structured_then_unstructured | 검색 경로 선택 |
| groups | array[array[Requirement]] | 그룹 사이 OR, 그룹 내부 AND. 각 AND를 같은 카드가 충족 |
| card_keys | array[string] | 공급한 catalog의 정확한 키. []는 전체 범위. 스키마 enum이 아니라 후속 목록 검사 |
| unresolved | array[string] | 미지원/불명확 조건. 비어 있지 않으면 현재 실험에서 복귀 |
| Requirement.targets | array[string] | 동일 혜택의 OR 대상. []는 대상 제한 없음 |
| Requirement.action | discount / earn / any | 할인 / 적립 계열 / 종류 제한 없음 |
| Requirement.min_percent | float 또는 null | percent >= 요구값. 정확히 같은 비율/최대 비율 필드 없음 |
| Requirement.max_spend_krw | int 또는 null | 필요 전월 최소 실적 <= 사용자가 허용한 상한. 개인 소비 예산을 자동 대입하지 않음 |
| Requirement.min_cap_krw | int 또는 null | 지급 한도 >= 요구 금액 |
| Requirement.cap_period | month / day / any | min_cap_krw가 있을 때만 기간 검사. 기간만 지정한 조건은 현재 별도 필터가 안 됨 |
| Requirement.exclude_targets | array[string] | 동일 fact 대상 문자열에 제외 대상이 있으면 해당 fact 미충족 |

스키마에 리스트 최소 길이·숫자 최소/최대 범위 제약이 없다. 예를 들어 groups=[[]]도 현재 형식 검사에 통과한다. 숫자 문자열 "10"은 Pydantic에서 수치로 정규화되지만 "10%"는 실패한다. 음수/과도한 수치의 의미적 유효성이 이 스키마로 보장되는 것은 아니다.

### 같은 혜택 관계와 실행 방식

```text
하나의 Requirement: 커피 + 할인 + 최소 10%
→ 같은 fact에서 대상·행동·비율·요구 실적·요구 한도 확인

AND 그룹: [커피 10% 할인, 넷플릭스 10% 할인]
→ 같은 카드의 각 조건을 충족하는 fact가 필요
→ 서로 다른 혜택 fact로 각 조건을 충족해도 됨

OR 그룹: [[커피 10% 할인], [넷플릭스 10% 할인]]
→ 어느 한 그룹을 충족하면 후보
```

현재 DB 읽기는 documents와 facts를 JOIN하여 raw_json/derived_json을 복원하고 Python 선택기가 비교한다. SQL 컬럼 인덱스로 조건별 WHERE를 생성하는 방식이 아니다. 상태가 true인 그룹의 fact_id를 카드별 matching_fact_ids로 반환한다. unknown은 true가 없고 필요한 값이 불확실한 카드로 분리되며, 하나라도 true 그룹이 있으면 후보가 된다.

```json
{
  "route": "structured_then_unstructured",
  "groups": [[{
    "targets": ["커피"], "action": "discount", "min_percent": 10,
    "max_spend_krw": null, "min_cap_krw": null, "cap_period": "any",
    "exclude_targets": []
  }]],
  "card_keys": [], "unresolved": []
}
```

위 조건과 일치한 카드 및 fact를 선택한다. 현재 S_THEN_U는 카드 목록만 검색 범위로 넘겨 원래 질문으로 벡터+BM25 검색한다. matching_fact_ids가 가리키는 원문 근거는 아직 최종 검색 후보에 자동 추가하지 않는다.




In [4]:
print(json.dumps(fresh['verification'], ensure_ascii=False, indent=2))
print('실험 API 요청:', fresh['new_provider_requests'])
print('추가 실험 API 비용(USD):', fresh['new_experiment_api_cost_usd'])
print('운영 변경:', fresh['operating_changes'])
print('검색 방식 성능 비교 실행:', fresh['performance_comparison_executed'])
display(Markdown(document[document.index('## 12. '):]))

{
  "integrity_check": "ok",
  "foreign_key_errors": [],
  "sqlite_json_fact_equality_rows": 3403,
  "sqlite_json_document_equality_rows": 106,
  "build_summary_artifact_hashes_match": true,
  "input_hash_size_mtime_unchanged": true
}
실험 API 요청: 0
추가 실험 API 비용(USD): 0
운영 변경: False
검색 방식 성능 비교 실행: False


## 12. 지원 범위·빈 곳·다음 비교에 필요한 확인

| 요구/정보 | 현재 상태 | 후속 확인 |
| --- | --- | --- |
| 가맹점/업종·할인/적립·최소 비율 | 단일 fact 조건 지원 | 고정 별칭/문자열 포함 검사의 오탐·미탐 |
| 전월 최소 실적·지급 한도·월/일 | known인 단일 값만 지원 | 복수 실적 구간·공유 한도·혼합 기간 처리 |
| 원화 정액 할인·마일리지·포인트 환산 | 원문/typed로 보존하지만 QueryPlan 비교 필드 없음 | 필요 질문을 먼저 정하고 단위별 필터 설계 |
| 횟수/행사 날짜/결제 채널/연회비/개인 예산 최적화 | 일부 원문 정보는 보존. 전용 필터 없음 | 비구조화 보완 필요. raw 저장과 검색 지원을 구분 |
| 전체 카드 기준 제외 혜택 | exclude_targets는 개별 fact 대상에 적용 | 카드의 다른 혜택에 제외 대상이 있으면 카드 전체를 제외하는 정책과는 다름 |
| 긍정 혜택과 적용 제외 관계 | action 부정/예외 대상은 보수 처리 | 전체 관계 의미 검증 및 OCR 검증은 미실행 |
| fact → 원문 → 검색 chunk | 원문 fragment/줄/범위 있음, chunk_id SQL 컬럼 및 매핑 테이블 없음 | 다음 비교 전에 원문 연결·같은 카드 소유권·복수 청크 연결 검증 |
| 구조화 후보가 남는 의미 오류 | 형식 검사만으로 검출되지 않음 | 질문-조건 의미 검증/재검색 실험 필요 |
| 추출 실패의 지정 카드 범위 | 현재 예외 처리에서 card_keys 초기화 | 신뢰 가능한 지정 범위 보존 방식을 실험할 필요 |
| 빈 내부 AND 그룹 | 현재 모든 범위 카드를 통과시킬 수 있음 | 합의 방향은 빈 조건을 사용하지 않고 비구조화 복귀. 코드에는 아직 미반영 |

fact_id는 내용 전체의 해시가 아니다. 같은 문서/PDF/순서에서 fact 문구가 바뀌어도 fact_id만으로 변경을 알아낼 수 없으므로 normalized_json_sha256와 artifact 해시를 함께 확인한다. 구조화 후보 선택이 정답 라벨을 대신하지 않으며 OCR 통과 가정과 제목 계층 미완성도 유지한다.

## 13. 검증·자원·문서 관리

실제 SQLite를 mode=ro&immutable=1 및 query_only=ON으로 읽었다. 입력 DB/JSONL/요약/참고 코드의 해시·크기·mtime 보존, summary artifact SHA 일치, integrity_check=ok, foreign_key_check 오류 없음, SQLite와 JSONL facts 3,403행 및 documents 106행의 일치를 확인했다. 불변 모드는 WAL을 무시하므로 현재 정적 실험 DB에 적용했으며 활성 writer가 있는 운영 DB의 최신 조회 방법으로 일반화하지 않는다.

새 실험 API 요청 0, 추가 실험 API 사용량 비용 0달러, 새 임베딩/OCR/모델 검색 추론 0. Codex 문서 작성의 모델 사용료·청구액, 경과/CPU/GPU/최대 메모리는 미측정이다. 다음 방식 비교의 성능·비용·지연은 아직 측정하지 않았다. 정적 HTML과 로컬 링크는 확인하며 브라우저 시각 검증 여부는 별도 완료 기록에 적는다.

```bash
python -B notebooks/44_inspect_structured_db.py
```

명세는 docs/STRUCTURED_SEARCH_DB_SPEC.md와 docs/STRUCTURED_SEARCH_DB_SPEC.html에서 관리한다. DB 구성이나 값 규칙이 바뀌면 실제 DB·생성 코드와 대조해 두 문서를 함께 갱신한다. 전용 문서 생성 코드는 제거했다. 기존 44번 inspector와 노트북은 DB 스키마·저장값 확인용으로만 유지하며, inspector는 원본 DB를 재생성하지 않고 JSON inventory만 작성한다. 노트북은 저장 inventory를 읽기 전용으로 재검사하고 이 명세를 표시한다. 전체 유료 실험을 재실행하지 않는다.

## 최종 정리

현재 4테이블 41컬럼의 선언·값 규칙·관측 분포·JSON 내부 근거·질문 조건 연결을 문서화했다. 원문과 파생값의 관계는 유지되지만 원화 정액·복합 구간 등 검색 미지원 정보가 남아 있다. 이 명세를 작성·정리하면서 운영/DB/이전 비교 실험의 검색 설정은 변경하지 않았다. 추가 실험 API 비용 0, 시간·메모리 미측정이다. 다음 단계는 fact와 원문 청크를 연결해 현재 카드 범위 검색과 근거 보완 검색을 같은 조건에서 비교하는 것이다. 성능 우열은 아직 미확정이다.
